In [ ]:
!pip install unsloth
!pip install --no-deps "xformers<0.0.28" "trl<0.10.0" peft accelerate bitsandbytes
!pip install qwen-vl-utils

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [2]:
import json
import os
from datasets import Dataset

# 1. Setup your paths (Make sure these match your workspace setup)
WORKSPACE_DIR = "/workspace/ImageToSpec_Stage1"
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "final_balanced_images")
TEST_DATASET_PATH = os.path.join(WORKSPACE_DIR, "qwen25_vl_3b_test.json") 

# 2. Define the loading function
def load_eval_dataset(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_data = []
    for item in data:
        # Clean up the image paths to point to your local directory
        raw_img_path = item["images"][0].replace("\\", "/")
        img_filename = os.path.basename(raw_img_path)
        local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)
        
        # Extract the target JSON string for scoring later
        target_spec_str = item["messages"][1]["content"] if len(item["messages"]) > 1 else ""

        processed_data.append({
            "id": str(item.get("id", "unknown")),
            "messages": item["messages"],
            "images": [local_img_path],
            "target_specs": target_spec_str
        })
    return Dataset.from_list(processed_data)

# 3. Load it into the variable!
eval_dataset = load_eval_dataset(TEST_DATASET_PATH)
print(f"Success! Loaded {len(eval_dataset)} samples into eval_dataset.")

Success! Loaded 260 samples into eval_dataset.


In [3]:
# ==========================================
# 2. REWARD FUNCTIONS
# ==========================================

def format_reward_func(completions, **kwargs):
    """Reward 1: Does the output compile as valid JSON? (The Gatekeeper)"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        # Heavy penalty for breaking syntax; this is a baseline requirement.
        rewards.append(1.0 if parsed is not None else -2.0)
    return rewards


def schema_enforcement_reward_func(completions, **kwargs):
    """Reward 2: Are the top-level keys correct and forbidden keys absent?"""
    rewards = []
    required_keys = {"title", "panel_count",
                     "panel_layout", "panels", "chart_type"}
    forbidden_keys = {"math", "rel", "stats", "trend"}

    for comp in completions:
        parsed = extract_json(comp)
        if not parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        # +0.5 for hitting all required root keys
        if required_keys.issubset(parsed.keys()):
            score += 0.5

        # -1.0 penalty if the model hallucinates legacy token-heavy semantic fields
        comp_str = str(comp).lower()
        if any(f'"{fk}"' in comp_str for fk in forbidden_keys):
            score -= 1.0

        rewards.append(score)
    return rewards


def panel_architecture_reward_func(completions, **kwargs):
    """Reward 3: Does the panel math make logical sense?"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        if not parsed or "panels" not in parsed or "panel_count" not in parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        try:
            # Does the stated count match the actual array length?
            actual_count = len(parsed["panels"])
            if actual_count == parsed["panel_count"]:
                score += 0.5

            # Does the layout grid support the panel count? (e.g., [2,1] grid supports up to 2 panels)
            if "panel_layout" in parsed and len(parsed["panel_layout"]) == 2:
                grid_capacity = parsed["panel_layout"][0] * \
                    parsed["panel_layout"][1]
                if grid_capacity >= actual_count:
                    score += 0.5
        except (TypeError, ValueError):
            pass

        rewards.append(score)
    return rewards


def axis_mapping_reward_func(completions, **kwargs):
    """Reward 4: Are axes structurally sound before checking series data?"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        if not parsed or "panels" not in parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        total_panels = len(parsed["panels"])
        valid_panels = 0

        for panel in parsed["panels"]:
            if "axes" in panel and isinstance(panel["axes"], list):
                # Check if at least an x_axis and y_axis are defined properly
                axes_names = [ax.get("name")
                              for ax in panel["axes"] if isinstance(ax, dict)]
                if "x_axis" in axes_names and "y_axis" in axes_names:
                    valid_panels += 1

        # Normalize score based on how many panels have valid axis definitions
        if total_panels > 0:
            score += (valid_panels / total_panels) * 1.0

        rewards.append(score)
    return rewards


def evaluate_bar_series(parsed_panel, target_panel):
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # A. Check Categorical X
            if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                data_score += 0.2

            # B. Check Primary Y Value
            try:
                p_val, t_val = robust_float(p_pt[1]), robust_float(t_pt[1])
                tol = max(0.05 * abs(t_val), 0.01)
                if abs(p_val - t_val) <= tol:
                    data_score += 0.5
                elif abs(p_val - t_val) <= tol * 2:
                    data_score += 0.2
            except ValueError:
                pass

            # C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2)
            for i in range(2, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_val_err, t_val_err = robust_float(
                        p_pt[i]), robust_float(t_pt[i])
                    tol_err = max(0.05 * abs(t_val_err), 0.01)
                    if abs(p_val_err - t_val_err) <= tol_err:
                        data_score += 0.25
                except ValueError:
                    # Fallback: It's a text marker (e.g., "Point A")
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += 0.25

        score += data_score / max(1, len(t_data))
    return score


def evaluate_line_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Line and Area charts (Standard, Stacked, Errorband).
    Dynamically handles both 2D [x, y] coordinates and 3D [x, y_max, y_min] boundaries.
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward: Series count match
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Points
    for p_s, t_s in zip(parsed_ser, target_ser):

        # Reward correct series mapping
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # --- A. Check X-Axis Value (Index 0: Hybrid Categorical/Numerical) ---
            try:
                # Attempt numerical matching first
                p_x = robust_float(p_pt[0])
                t_x = robust_float(t_pt[0])
                tol_x = max(0.05 * abs(t_x), 0.01)

                if abs(p_x - t_x) <= tol_x:
                    data_score += 0.2
            except (ValueError, TypeError):
                # Fallback to categorical string matching (e.g., "January", "QuickSort")
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    data_score += 0.2

            # --- B. Check Primary Y-Axis Value (Index 1) ---
            try:
                p_y = robust_float(p_pt[1])
                t_y = robust_float(t_pt[1])
                tol_y = max(0.05 * abs(t_y), 0.01)

                if abs(p_y - t_y) <= tol_y:
                    data_score += 0.5
                elif abs(p_y - t_y) <= tol_y * 2:
                    data_score += 0.2  # Partial credit for being close
            except (ValueError, TypeError):
                pass

            # --- C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2) ---
            for i in range(2, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_val_extra, t_val_extra = robust_float(
                        p_pt[i]), robust_float(t_pt[i])
                    tol_extra = max(0.05 * abs(t_val_extra), 0.01)
                    if abs(p_val_extra - t_val_extra) <= tol_extra:
                        data_score += 0.3
                    elif abs(p_val_extra - t_val_extra) <= tol_extra * 2:
                        data_score += 0.1
                except ValueError:
                    # Fallback: It's a text marker (e.g., "Point A")
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += 0.3

        # Normalize to prevent massive arrays from disproportionately skewing the RL objective
        normalized_data_score = data_score / max(1, len(t_data))
        score += normalized_data_score

    return score


def evaluate_box_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Box plots.
    Dynamically handles both 5-number summary arrays (length 6) and individual outlier points (length 2).
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward: Series count match (Usually 1 for summary, 1 for outliers)
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Points
    for p_s, t_s in zip(parsed_ser, target_ser):

        # Reward correct series mapping ("Box Distribution" vs "Outliers")
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # Reward finding the outlier flag correctly
        if t_s.get("is_outlier") == 1 and p_s.get("is_outlier") == 1:
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # --- A. Check Categorical X-Axis Value (Index 0) ---
            try:
                # Attempt numerical matching first (rare for boxplots, but safe)
                p_x = robust_float(p_pt[0])
                t_x = robust_float(t_pt[0])
                tol_x = max(0.05 * abs(t_x), 0.01)

                if abs(p_x - t_x) <= tol_x:
                    data_score += 0.2
            except (ValueError, TypeError):
                # Fallback to categorical string matching (e.g., "E.coli")
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    data_score += 0.2

            # --- B. Dynamically Check Y-Axis Values (Indices 1 through End) ---
            # If t_pt has length 6 (5-num summary), each stat is worth 0.2 (total 1.0)
            # If t_pt has length 2 (outlier), the single stat is worth 1.0
            # --- B. Dynamically Check Y-Axis Values (Indices 1 through End) ---
            stat_weight = 1.0 / (len(t_pt) - 1)

            for i in range(1, len(t_pt)):
                if i >= len(p_pt):
                    break

                try:
                    p_val, t_val = robust_float(p_pt[i]), robust_float(t_pt[i])
                    tol_val = max(0.05 * abs(t_val), 0.01)

                    if abs(p_val - t_val) <= tol_val:
                        data_score += stat_weight
                    elif abs(p_val - t_val) <= tol_val * 2:
                        data_score += stat_weight * 0.4
                except ValueError:
                    # Fallback for text markers in outlier arrays (e.g. [Cat, Value, "Outlier A"])
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += stat_weight

        # Normalize the data score by the target array length.
        normalized_data_score = data_score / max(1, len(t_data))
        score += normalized_data_score

    return score


def evaluate_histogram_series(parsed_panel, target_panel):
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Structures
    for p_s, t_s in zip(parsed_ser, target_ser):
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # --- PATH A: KDE Density Curves ---
        if t_s.get("is_kde") == 1 or t_s.get("type") == "density_curve":
            if p_s.get("is_kde") == 1:
                score += 0.1

            t_peak, p_peak = t_s.get(
                "exact_peak", {}), p_s.get("exact_peak", {})
            if isinstance(t_peak, dict) and isinstance(p_peak, dict) and t_peak:
                try:
                    for k in ["x", "y"]:
                        p_val, t_val = robust_float(p_peak.get(
                            k, 0)), robust_float(t_peak.get(k, 0))
                        if abs(p_val - t_val) <= max(0.05 * abs(t_val), 0.01):
                            score += 0.2
                except ValueError:
                    pass

            # Route to "points" array
            p_data, t_data = p_s.get("points", []), t_s.get("points", [])

        # --- PATH B: Aggregated & Normal Bins ---
        else:
            if t_s.get("is_aggregated") == 1 and p_s.get("is_aggregated") == 1:
                score += 0.1

            # Route to "data" array
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])

        # 3. Evaluate the Numerics
        if not p_data or not t_data:
            continue

        # ---------------------------------------------------------
        # THE ANTI-LOOPING & TRUNCATION PENALTY
        # Punish the model for missing points or hallucinating extra points.
        # This works perfectly for both 25-bin arrays and 150-point KDE curves.
        # ---------------------------------------------------------
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            data_score -= (0.1 * length_diff)

        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            pt_weight = 1.0 / len(t_pt)

            for i in range(len(t_pt)):
                if i >= len(p_pt):
                    break

                try:
                    p_val, t_val = robust_float(p_pt[i]), robust_float(t_pt[i])
                    tol_val = max(0.05 * abs(t_val), 0.01)

                    if abs(p_val - t_val) <= tol_val:
                        data_score += pt_weight
                    elif abs(p_val - t_val) <= tol_val * 2:
                        data_score += pt_weight * 0.4
                except ValueError:
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += pt_weight

        score += data_score / max(1, len(t_data))

    return score


def evaluate_scatter_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Scatter plots.
    Dynamically handles standard [x, y] coordinates and heavily compressed 
    k-means cluster descriptions (centroids, bounding boxes, outliers).
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        # 1. Base Metadata
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # Reward finding the correct secondary axis mapping (Crucial for multi-axes charts)
        if str(p_s.get("y_ref")) == str(t_s.get("y_ref")):
            score += 0.2

        # ---------------------------------------------------------
        # PATH A: Clustered Scatter (Spatial & Density Math)
        # ---------------------------------------------------------
        if t_s.get("is_clustered") == 1:
            if p_s.get("is_clustered") == 1:
                score += 0.2

            t_clusters = t_s.get("cluster_descriptions", [])
            p_clusters = p_s.get("cluster_descriptions", [])

            cluster_score = 0.0
            for p_c, t_c in zip(p_clusters, t_clusters):
                if not isinstance(p_c, dict) or not isinstance(t_c, dict):
                    continue

                # A1. Score Centroids
                try:
                    p_cx, t_cx = robust_float(p_c.get("centroid", [0, 0])[0]), robust_float(
                        t_c.get("centroid", [0, 0])[0])
                    p_cy, t_cy = robust_float(p_c.get("centroid", [0, 0])[1]), robust_float(
                        t_c.get("centroid", [0, 0])[1])

                    if abs(p_cx - t_cx) <= max(0.05 * abs(t_cx), 0.01):
                        cluster_score += 0.2
                    if abs(p_cy - t_cy) <= max(0.05 * abs(t_cy), 0.01):
                        cluster_score += 0.2
                except (ValueError, TypeError, IndexError):
                    pass

                # A2. Score Bounding Boxes
                p_bb, t_bb = p_c.get("bounding_box", {}), t_c.get(
                    "bounding_box", {})
                for k in ["x_min", "x_max", "y_min", "y_max"]:
                    try:
                        if abs(robust_float(p_bb.get(k)) - robust_float(t_bb.get(k))) <= max(0.05 * abs(robust_float(t_bb.get(k))), 0.01):
                            cluster_score += 0.1
                    except (ValueError, TypeError):
                        pass

                # A3. Score Density Metadata (Critical for Stage 2 Claims)
                try:
                    if abs(robust_float(p_c.get("density_percentage")) - robust_float(t_c.get("density_percentage"))) <= 1.0:
                        cluster_score += 0.2
                except (ValueError, TypeError):
                    pass

            score += cluster_score / max(1, len(t_clusters))

            # For clustered charts, the discrete points are stored in "outliers"
            p_data = p_s.get("outliers", [])
            t_data = t_s.get("outliers", [])

        # ---------------------------------------------------------
        # PATH B: Standard Scatter (Inside evaluate_scatter_series)
        # ---------------------------------------------------------
        else:
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])
            if not p_data or not t_data:
                continue

            data_score = 0.0
            for p_pt, t_pt in zip(p_data, t_data):
                if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                    continue

                # Check X
                try:
                    if abs(robust_float(p_pt[0]) - robust_float(t_pt[0])) <= max(0.05 * abs(robust_float(t_pt[0])), 0.01):
                        data_score += 0.2
                except ValueError:
                    if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                        data_score += 0.2

                # Check Y
                try:
                    if abs(robust_float(p_pt[1]) - robust_float(t_pt[1])) <= max(0.05 * abs(robust_float(t_pt[1])), 0.01):
                        data_score += 0.5
                    elif abs(robust_float(p_pt[1]) - robust_float(t_pt[1])) <= max(0.10 * abs(robust_float(t_pt[1])), 0.02):
                        data_score += 0.2
                except ValueError:
                    pass

                # --- C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2) ---
                for i in range(2, len(t_pt)):
                    if i >= len(p_pt):
                        break
                    try:
                        p_val_extra, t_val_extra = robust_float(
                            p_pt[i]), robust_float(t_pt[i])
                        tol_extra = max(0.05 * abs(t_val_extra), 0.01)
                        if abs(p_val_extra - t_val_extra) <= tol_extra:
                            data_score += 0.2
                    except ValueError:
                        # Fallback: It's a text marker
                        if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                            data_score += 0.2

            score += data_score / max(1, len(t_data))

    return score


def evaluate_pie_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Pie and Donut charts.
    Evaluates exact percentage extractions with tight tolerances, 
    and optional absolute value pairings.
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        # Match the slice label (often stored in the 'name' field)
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        # Pie charts usually have a single inner array per series definition: data[0]
        p_pt = p_data[0] if isinstance(p_data[0], list) else []
        t_pt = t_data[0] if isinstance(t_data[0], list) else []

        if not p_pt or not t_pt:
            continue

        # 1. Verify Percentage (Index 0)
        # Using a tighter 2% tolerance here since slices are highly dependent on exact ratios
        try:
            p_pct = robust_float(p_pt[0])
            t_pct = robust_float(t_pt[0])
            tol_pct = max(0.02 * abs(t_pct), 0.005)

            if abs(p_pct - t_pct) <= tol_pct:
                score += 0.5
            elif abs(p_pct - t_pct) <= tol_pct * 2:
                score += 0.2
        except (ValueError, TypeError):
            pass

        # 2. Verify Absolute Value if present (Index 1)
        if len(t_pt) > 1 and len(p_pt) > 1:
            try:
                p_val = robust_float(p_pt[1])
                t_val = robust_float(t_pt[1])
                tol_val = max(0.05 * abs(t_val), 0.01)

                if abs(p_val - t_val) <= tol_val:
                    score += 0.5
            except (ValueError, TypeError):
                pass

    # Normalize score across the number of pie slices
    return score / max(1, len(target_ser))

In [ ]:
import torch
from unsloth import FastVisionModel
from tqdm import tqdm
import PIL.Image

# 1. Load the Model AND the LoRA Adapter simultaneously!
# Passing the adapter path directly bypasses the PEFT load_adapter bug.
# Unsloth will read the config, fetch the base Qwen model, and apply the weights.
model, processor = FastVisionModel.from_pretrained(
    "/workspace/ImageToSpec_Stage1/qwen3b_lora_sft_bf16_final/qwen3b_lora_sft_bf16_final/",
    load_in_4bit=False,  # <--- CHANGED: RTX 3090 has 24GB VRAM! Run in native precision.
    use_gradient_checkpointing="unsloth",
)

# 2. Optimize for Inference
FastVisionModel.for_inference(model)

# 3. Increase Image Resolution Limits
# Changed from 262144 (512x512) to 1003520 (~1024x1024). 
# Your 3090 can easily handle this, and it will drastically improve data extraction accuracy!
# processor.image_processor.max_pixels = 1003520 

all_completions = []
all_targets = []

print("Running Native Precision Inference on RTX 3090...")

for item in tqdm(eval_dataset):
    messages = item["messages"]

    # We only feed the user prompt to the model (messages[0])
    prompt = processor.apply_chat_template(
        [messages[0]], tokenize=False, add_generation_prompt=True)

    # Load the image using PIL
    image_path = item["images"][0]
    pil_image = PIL.Image.open(image_path).convert("RGB")

    # Format inputs for Qwen-VL
    inputs = processor(
        text=[prompt],
        images=[pil_image],
        return_tensors="pt",
        padding=True
    ).to("cuda")

    # Generate completion
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=3072,  
            do_sample=False      
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    completion = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    all_completions.append(completion)
    all_targets.append(item["target_specs"])




In [8]:
import re
# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================
def extract_json(text):
    """Safely extracts JSON from model completions that might include markdown."""
    text = text.strip()
    json_match = re.search(r'```json\n(.*?)\n```', text, re.DOTALL)
    if json_match:
        text = json_match.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None
    

def robust_float(val):
    """
    Safely casts strings, integers, and scientific notation (3e+12) to floats.
    Strips LLM hallucinations like stray spaces or commas.
    """
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        try:
            # Clean common LLM formatting artifacts
            clean_val = val.strip().replace(',', '')
            # Python handles "3e+12" and "-2.45e-10" natively
            return float(clean_val)
        except ValueError:
            raise ValueError(f"Cannot parse {val} to float")
    raise ValueError("Not a valid type for float conversion")


def dynamic_series_reward_func(completions, target_specs, **kwargs):
    """Reward 5: Master routing calculation for the `ser` array based on chart subtype."""
    rewards = []

    for comp, target_str in zip(completions, target_specs):
        parsed = extract_json(comp)
        target = target_str if isinstance(
            target_str, dict) else json.loads(target_str)

        if not parsed or "panels" not in parsed or "panels" not in target:
            rewards.append(0.0)
            continue

        overall_score = 0.0
        target_panel_count = len(target["panels"])

        for p_panel, t_panel in zip(parsed["panels"], target["panels"]):
            chart_type = t_panel.get("topo", {}).get("type", "")
            panel_score = 0.0

            if chart_type == "bar":
                panel_score = evaluate_bar_series(p_panel, t_panel)
            elif chart_type == "line":
                panel_score = evaluate_line_series(p_panel, t_panel)
            elif chart_type == "box":
                panel_score = evaluate_box_series(p_panel, t_panel)
            elif chart_type == "histogram":
                panel_score = evaluate_histogram_series(p_panel, t_panel)
            elif chart_type == "scatter":
                panel_score = evaluate_scatter_series(p_panel, t_panel)
            elif chart_type == "pie":
                panel_score = evaluate_pie_series(p_panel, t_panel)

            overall_score += panel_score

        # NORMALIZE BY PANEL COUNT:
        # Ensures multi-panel charts have the same maximum reward ceiling (1.0) as single-panel charts
        normalized_total_score = overall_score / max(1, target_panel_count)
        rewards.append(normalized_total_score)

    return rewards

# ==========================================
# 4. Score the completions
# ==========================================
print("\n=== EVALUATION RESULTS ===")

format_scores = format_reward_func(all_completions)
print(f"Format Accuracy:         {sum(1 for s in format_scores if s > 0) / len(format_scores) * 100:.2f}%")

schema_scores = schema_enforcement_reward_func(all_completions)
print(f"Avg Schema Reward:       {sum(schema_scores)/len(schema_scores):.4f}")

arch_scores = panel_architecture_reward_func(all_completions)
print(f"Avg Architecture Reward: {sum(arch_scores)/len(arch_scores):.4f}")

axis_scores = axis_mapping_reward_func(all_completions)
print(f"Avg Axis Reward:         {sum(axis_scores)/len(axis_scores):.4f}")

data_scores = dynamic_series_reward_func(all_completions, all_targets)
print(f"Avg Series Data Reward:  {sum(data_scores)/len(data_scores):.4f}")


#save the completions in json
import json
print("Saving completions to JSON...")

# Package the inputs, targets, and generations together so you can easily review them
results_to_save = []
for i, item in enumerate(eval_dataset):
    results_to_save.append({
        "id": item.get("id", f"sample_{i}"),
        "image_path": item["images"][0],
        "target_spec": all_targets[i],
        "generated_completion": all_completions[i]
    })

# Save to your workspace
output_filepath = "/workspace/qwen3b_inference_results.json"
with open(output_filepath, "w", encoding="utf-8") as f:
    json.dump(results_to_save, f, indent=4)

print(f"Successfully saved {len(results_to_save)} completions to {output_filepath}")


=== EVALUATION RESULTS ===
Format Accuracy:         24.23%
Avg Schema Reward:       0.1135
Avg Architecture Reward: 0.2423
Avg Axis Reward:         0.2423
Avg Series Data Reward:  0.5665
Saving completions to JSON...
Successfully saved 260 completions to /workspace/qwen3b_inference_results.json
